# Academic Advisor: model training
Predicts how a student will perform in a course (At Risk / Average / Strong)
from their history and habits. Trained on `model_ready.csv`.

Run this on Google Colab. Steps: load data, split by student, compare models,
evaluate on a held-out set, then save one `.pkl` that contains preprocessing + model.

In [ ]:
import json
import os
import subprocess
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
print("scikit-learn", sklearn.__version__)

## 1. Load the data
If you pushed the repo to GitHub, set `REPO_URL` and the notebook clones it.
Otherwise it asks you to upload `model_ready.csv`.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/academic-advisor.git"

CANDIDATES = ["data/model_ready.csv", "academic-advisor/data/model_ready.csv"]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if path is None and "google.colab" in sys.modules and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", REPO_URL], check=False)
    path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if path is None:
    try:
        from google.colab import files
        print("Upload model_ready.csv")
        path = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError("model_ready.csv not found")

df = pd.read_csv(path)
print(df.shape)
df.head()

In [ ]:
print(df["grade_band"].value_counts(normalize=True).round(3))
print("missing values:", int(df.isna().sum().sum()))

## 2. Features and target
`score` and `grade` are the answer in another form, so they are dropped (leakage).
`student_id` and `course_code` are identifiers. `student_id` is kept only to split by student.

In [ ]:
TARGET = "grade_band"
DROP = ["student_id", "course_code", "score", "grade", TARGET]

X = df.drop(columns=DROP)
y = df[TARGET]
groups = df["student_id"]

CATEGORICAL = ["category"]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]
print("numeric:", NUMERIC)

## 3. Train/test split by student
The same student must never appear in both sets, otherwise the test score is inflated.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]
print(len(X_train), "train rows /", len(X_test), "test rows")

## 4. Compare models with grouped cross-validation

In [ ]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=10,
                                            random_state=SEED, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05,
                                                        max_iter=200, random_state=SEED),
}

cv = GroupKFold(n_splits=5)
cv_scores = {}
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", clf)])
    s = cross_val_score(pipe, X_train, y_train, groups=g_train, cv=cv,
                        scoring="f1_macro", n_jobs=-1)
    cv_scores[name] = s.mean()
    print(f"{name:20s} macro F1 = {s.mean():.3f} (+/- {s.std():.3f})")

best_name = max(cv_scores, key=cv_scores.get)
print("\nBest model:", best_name)

## 5. Fit the best model and evaluate on the held-out students

In [ ]:
pipeline = Pipeline([("prep", preprocess), ("model", candidates[best_name])])
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="macro")
print(f"accuracy = {acc:.3f}   macro F1 = {f1:.3f}\n")
print(classification_report(y_test, pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title(f"{best_name}: confusion matrix (held-out students)")
plt.tight_layout()
plt.show()

## 6. What does the model rely on?

In [ ]:
sample = X_test.sample(min(3000, len(X_test)), random_state=SEED)
imp = permutation_importance(pipeline, sample, y_test.loc[sample.index],
                             n_repeats=5, scoring="f1_macro", random_state=SEED, n_jobs=-1)
order = np.argsort(imp.importances_mean)
plt.figure(figsize=(7, 6))
plt.barh(np.array(X.columns)[order], imp.importances_mean[order], color="#3b82f6")
plt.xlabel("Drop in macro F1 when the feature is shuffled")
plt.title("Permutation importance")
plt.tight_layout()
plt.show()

## 7. Save the model (.pkl) and its metadata
The pickle holds the whole pipeline (encoding, scaling and model), so the web app
only needs to pass in a table of raw feature values.

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(pipeline, "models/advisor_model.pkl")

metadata = {
    "model": best_name,
    "classes": list(pipeline.classes_),
    "features": list(X.columns),
    "categorical": CATEGORICAL,
    "numeric": NUMERIC,
    "metrics": {"accuracy": round(acc, 4), "macro_f1": round(f1, 4),
                "cv_macro_f1": {k: round(v, 4) for k, v in cv_scores.items()}},
    "sklearn_version": sklearn.__version__,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
}
with open("models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))
print(f"\nPin this in requirements.txt:  scikit-learn=={sklearn.__version__}")

## 8. Quick sanity check: load the pickle back and predict

In [ ]:
loaded = joblib.load("models/advisor_model.pkl")
row = X_test.iloc[[0]]
print(row.T)
print("prediction:", loaded.predict(row)[0])
print(dict(zip(loaded.classes_, loaded.predict_proba(row)[0].round(3))))

In [ ]:
# Colab only: download the two files
try:
    from google.colab import files
    files.download("models/advisor_model.pkl")
    files.download("models/model_metadata.json")
except ImportError:
    pass